In [1]:
from ATLAS.Registration.execute import *
from ATLAS.Registration.execute import *
from scipy.ndimage import grey_erosion, grey_dilation
from ATLAS.Utils import geomu
from ATLAS.Utils import basicu
from ATLAS.Analysis.Classification import *
from scipy.interpolate import interp1d
np.random.seed(42)
adata = anndata.read_h5ad('/scratchdata1/ExternalData/Zhaung_WMB/WB_imputation_animal1_coronal_anterior_treeDPNMF.h5ad')
adata.obs['true_subclass'] = adata.obs['subclass'].copy()
adata.layers['raw'] = adata.X.copy()
adata

In [4]:
np.random.seed(42)
self = SingleCellAlignmentLeveragingExpectations(adata,visualize=False,verbose=True)
self.likelihood_only = False
# self.calculate_spatial_priors()
self.update_user("Loading Spatial Priors Class")
kdesp = KDESpatialPriors(ref_levels=[self.ref_level],neuron=None,kernel=(0.25,0.1,0.1))
kdesp.train()

max_magnitude = 5 # *0.1 # 0.1 is the voxel size in mm
sizes = np.random.randint(-max_magnitude,max_magnitude,size=kdesp.types.shape[0])
typedata = kdesp.typedata.copy()
for i,size in tqdm(enumerate(sizes),desc='Applying random erosion/dilation',total=kdesp.types.shape[0]):
    single_type_3d = kdesp.typedata[:,:,:,i].copy().astype(np.float32)
    single_type_3d = grey_dilation(single_type_3d, size=(max_magnitude,max_magnitude,max_magnitude))
    # single_type_3d = grey_erosion(single_type_3d, size=(max_magnitude,max_magnitude,max_magnitude))
    single_type_3d = gaussian_filter(single_type_3d,sigma=2.5)
    typedata[:,:,:,i] = single_type_3d.astype(np.float16)
kdesp.typedata = typedata
self.update_user("Calculating Spatial priors")
priors = {}
priors,types = kdesp.classify(self.measured, level=self.ref_level,dim_labels=['ccf_x','ccf_y','ccf_z'])
priors[np.sum(priors,axis=1)==0,:] = 1 # if all zeros make it uniform
priors = {'columns':types,'indexes':np.array(self.measured.obs.index),'matrix':priors.astype(np.float32)}
self.priors = priors
self.load_reference()
self.model = LogisticRegression(max_iter=1000,random_state=42) 
self.supervised_neuron_annotation()

self.supervised_harmonization()
adata1 = self.measured.copy()

In [8]:
remove_index = {ct:ct[4:] for ct in adata1.obs['subclass'].unique()}
adata1.obs['predicted_subclass'] = adata1.obs['subclass'].map(remove_index)
adata1.obs[['subclass','predicted_subclass','true_subclass']]

In [9]:
adata1[adata1.obs['predicted_subclass']!=adata1.obs['true_subclass']].obs[['subclass','predicted_subclass','true_subclass','ccf_x']].head(20)

In [10]:
np.mean(adata1.obs['predicted_subclass']==adata1.obs['true_subclass'])

In [13]:
kdesp = KDESpatialPriors(ref_levels=[self.ref_level],neuron=None,kernel=(0.25,0.1,0.1))
kdesp.train()


In [209]:
# turn off fontTools logs
import logging
logging.getLogger('fontTools').setLevel(logging.ERROR)
ct = '058 PAL-STR Gaba-Chol'

i = [i for i,c in enumerate(kdesp.types) if ct in c][0]
single_type_3d = kdesp.typedata[:,:,:,i].copy().astype(np.float32)
img = single_type_3d.max(0)
vmin,vmax = np.percentile(img,[1,99])
vals = [-10,-5,0,5,10]
fig,axs = plt.subplots(1,len(vals),figsize=[5*len(vals),4],dpi=500)
axs = axs.ravel()
for i,val in enumerate(vals):
    ax = axs[i]
    temp = single_type_3d.copy()
    if val<0:
        label = f"{val*100} um \n erosion radius"
        size = int(-val)
        temp = grey_erosion(temp,size=(size,size,size))

    elif val>0:
        label = f"{val*100} um \n dilation radius"
        size = int(val)
        temp = grey_dilation(temp,size=(size,size,size))
    else:
        label = 'raw'
    temp = gaussian_filter(temp,sigma=2.5)
    img = 100*temp.max(0)
    vmin,vmax = np.percentile(img,[1,99])
    im = ax.imshow(img,vmin=vmin,vmax=vmax,cmap='inferno')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
    ax.set_title(label)
    ax.grid(False)
    # ax.axis('off')
    ax.axis('equal')
    ax.set_xlabel('ccf z (mm)')
    ax.set_ylabel('ccf y (mm)')
    ax.set_xticks(ax.get_xticks(),labels=[int(i) for i in ax.get_xticks()*0.1])
    ax.set_yticks(ax.get_yticks(),labels=[int(i) for i in ax.get_yticks()*0.1])
    ax.set_xlim([0,img.shape[1]])
    ax.set_ylim([img.shape[0],0])
plt.tight_layout()
plt.savefig(f"/scratchdata1/MouseBrainAtlases_V7/AnalysisNotebooks/Panels/Prior_{ct}_zy.pdf",dpi=500)
plt.show()

In [217]:
ct = '058 PAL-STR Gaba-Chol'

i = [i for i,c in enumerate(kdesp.types) if ct in c][0]
# mask = kdesp.typedata[:,:,:,:].sum(3).max(2)>0.1
single_type_3d = kdesp.typedata[:,:,:,i].copy().astype(np.float32)
img = single_type_3d.max(0)
vmin,vmax = np.percentile(img,[1,99])
vals = [-10,-5,0,5,10]
fig,axs = plt.subplots(1,len(vals),figsize=[5*len(vals),7],dpi=500)
axs = axs.ravel()
for i,val in enumerate(vals):
    ax = axs[i]
    temp = single_type_3d.copy()
    if val<0:
        label = f"{val*100} um \n erosion radius"
        size = int(-val)
        temp = grey_erosion(temp,size=(size,size,size))

    elif val>0:
        label = f"{val*100} um \n dilation radius"
        size = int(val)
        temp = grey_dilation(temp,size=(size,size,size))
    else:
        label = 'raw'
    temp = gaussian_filter(temp,sigma=2.5)
    img = 100*temp.max(2)
    img = img.T
    vmin,vmax = np.percentile(img[np.isnan(img)==False],[1,99])
    im = ax.imshow(img,vmin=vmin,vmax=vmax,cmap='inferno')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
    ax.set_title(label)
    ax.grid(False)
    ax.set_xlabel('ccf x (mm)')
    ax.set_ylabel('ccf y (mm)')
    ax.set_xticks(ax.get_xticks(),labels=[int(i) for i in ax.get_xticks()*0.1])
    ax.set_yticks(ax.get_yticks(),labels=[int(i) for i in ax.get_yticks()*0.1])
    ax.set_ylim([img.shape[0],0])
    ax.set_xlim([0,img.shape[1]])
plt.tight_layout()
plt.savefig(f"/scratchdata1/MouseBrainAtlases_V7/AnalysisNotebooks/Panels/Prior_{ct}_xy.pdf",dpi=500)
plt.show()

In [214]:
ct = '058 PAL-STR Gaba-Chol'

i = [i for i,c in enumerate(kdesp.types) if ct in c][0]
single_type_3d = kdesp.typedata[:,:,:,i].copy().astype(np.float32)
img = single_type_3d.max(0)
vmin,vmax = np.percentile(img,[1,99])
vals = [-10,-5,0,5,10]
fig,axs = plt.subplots(1,len(vals),figsize=[5*len(vals),5],dpi=500)
axs = axs.ravel()
for i,val in enumerate(vals):
    ax = axs[i]
    temp = single_type_3d.copy()
    if val<0:
        label = f"{val*100} um \n erosion radius"
        size = int(-val)
        temp = grey_erosion(temp,size=(size,size,size))

    elif val>0:
        label = f"{val*100} um \n dilation radius"
        size = int(val)
        temp = grey_dilation(temp,size=(size,size,size))
    else:
        label = 'raw'
    temp = gaussian_filter(temp,sigma=2.5)
    img = 100*temp.max(1)
    img = img.T
    vmin,vmax = np.percentile(img,[1,99])
    im = ax.imshow(img,vmin=vmin,vmax=vmax,cmap='inferno')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im, cax=cax)
    ax.set_title(label)
    ax.grid(False)
    ax.set_xlabel('ccf x (mm)')
    ax.set_ylabel('ccf z (mm)')
    ax.set_xticks(ax.get_xticks(),labels=[int(i) for i in ax.get_xticks()*0.1])
    ax.set_yticks(ax.get_yticks(),labels=[int(i) for i in ax.get_yticks()*0.1])
    ax.set_xlim([0,img.shape[1]])
    ax.set_ylim([0,img.shape[0]])
plt.tight_layout()
plt.savefig(f"/scratchdata1/MouseBrainAtlases_V7/AnalysisNotebooks/Panels/Prior_{ct}_xz.pdf",dpi=500)
plt.show()